# Session 1: slope is speed, area is distance
**A. Real-world problem.** A car's speedometer tells you how fast you go, but not how far you
have come. Its odometer tells you how far, but not how fast. Can one graph give you the other?

**Plan.** (1) describe a car by its speed at a few moments, (2) turn that into distance by
adding up areas, (3) turn distance back into speed by measuring slopes, (4) draw both graphs.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# The car starts at rest. Its speed (m/s) is known at t = 0, 2, 4, 6 seconds and is a straight
# line between those moments.
KNOT_T = np.array([0.0, 2.0, 4.0, 6.0])


def knot_speeds(a, b, c):
    """Speeds at the four moments: 0 at the start, then a, b, c (m/s)."""
    return np.array([0.0, a, b, c], dtype=float)

## Step 1: speed at any moment
Between two known moments the speed changes in a straight line. `np.interp` does exactly that.

In [ ]:
def speed(t, v_knots):
    """Speed (m/s) at time t (s), straight lines between the knots."""
    return np.interp(t, KNOT_T, v_knots)

## Step 2: distance = area under the speed graph
Each slice between two knots is a trapezoid: width (seconds) times the average of its two
parallel sides (m/s). Width * height has the unit s * m/s = m, a distance.
For a car at a steady 10 m/s for 2 s: area = 2 * 10 = 20 m.

In [ ]:
def distance_at_knots(v_knots):
    """Total distance (m) at each knot: running sum of trapezoid areas."""
    widths = np.diff(KNOT_T)
    avg = (v_knots[:-1] + v_knots[1:]) / 2
    return np.concatenate([[0.0], np.cumsum(widths * avg)])


def distance(t, v_knots):
    """Distance travelled (m) up to time t (s): whole slices plus a part of one slice."""
    t = np.asarray(t, dtype=float)
    dists = distance_at_knots(v_knots)
    k = np.clip(np.searchsorted(KNOT_T, t, side="right") - 1, 0, len(KNOT_T) - 2)
    dt = t - KNOT_T[k]
    v0 = v_knots[k]
    slope = (v_knots[k + 1] - v_knots[k]) / (KNOT_T[k + 1] - KNOT_T[k])
    return dists[k] + v0 * dt + 0.5 * slope * dt ** 2

## Step 3: speed = slope of the distance graph
The slope of a graph is "rise over run". On the distance graph the rise is metres and the run
is seconds, so the slope is m/s, a speed. We estimate it from two close moments.

In [ ]:
def slope_of_distance(t, v_knots, h=1e-4):
    """Slope (m/s) of the distance graph at time t, from two moments 2h apart."""
    return float((distance(t + h, v_knots) - distance(t - h, v_knots)) / (2 * h))

## Step 4: see it
Left: speed against time, with the area up to t = 4 s shaded. Right: distance against time;
the tangent line there has slope equal to the speed.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    v = knot_speeds(10.0, 14.0, 6.0)
    t = np.linspace(0, 6, 241)
    t0 = 4.0
    print("speeds at knots (m/s):", v)
    print("distance at knots (m):", distance_at_knots(v))
    print(f"distance at t = {t0} s: {float(distance(t0, v)):.2f} m")
    print(f"speed at t = {t0} s: {float(speed(t0, v)):.2f} m/s; slope of the distance graph there: {slope_of_distance(t0, v):.2f} m/s")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(t, speed(t, v), color="#1f77b4")
    tt = np.linspace(0, t0, 100)
    ax.fill_between(tt, speed(tt, v), alpha=0.3, label=f"area = {float(distance(t0, v)):.0f} m")
    ax.plot(KNOT_T, v, "o", color="#1f77b4")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("speed (m/s)")
    ax.set_title("Area under the speed graph is distance")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "speed_area.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(t, distance(t, v), color="#d62728", label="distance")
    sl = slope_of_distance(t0, v)
    d0 = float(distance(t0, v))
    ax2.plot([t0 - 1.5, t0 + 1.5], [d0 - 1.5 * sl, d0 + 1.5 * sl], "k--", label=f"tangent, slope {sl:.0f} m/s")
    ax2.plot([t0], [d0], "ko")
    ax2.set_xlabel("time (s)")
    ax2.set_ylabel("distance (m)")
    ax2.set_title("Slope of the distance graph is speed")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "distance_slope.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the speeds in `lesson.py` to 20, 20, 0 (a car that brakes to rest at 6 s). Predict the distance at each knot by hand, then run it.
2. **Stretch.** Write `average_speed(v_knots)` that returns the total distance divided by 6 s and check it for the speeds 0, 12, 12, 6 (answer 9 m/s).
3. **Explain.** Explain in your own words why the area under a speed graph is a distance, using the units of width and height.

## Assessment
- **A. Conceptual.** A friend says 'the speed graph is flat, so the car is not moving'. Is that right? What does a flat distance graph mean?
- **B. Numerical.** A car has speeds 0, 12, 12 and 6 m/s at t = 0, 2, 4 and 6 s, with straight lines between. Find the total distance in 6 s and the average speed. Check the units.
- **C. Graph and simulation interpretation.** In the distance graph of the car from the worked example (speeds 10, 14, 6 m/s), where is the curve steepest and what is its slope there? What is the slope at the end, t = 6 s?
- **D. Design and optimisation challenge.** Design: choose the speeds at 2, 4 and 6 s (each at most 20 m/s, the limit of this exercise) so that the car covers at least 90 m in 6 s. Find a choice that works and one that fails, and say what you would trade off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    steady = L.knot_speeds(10.0, 10.0, 10.0)
    checks.append(("steady 10 m/s for 2 s (t = 2 to 4) covers 20 m", abs(float(L.distance(4.0, steady) - L.distance(2.0, steady)) - 20.0) < 1e-9))
    checks.append(("start: distance 0 and speed 0", float(L.distance(0.0, steady)) == 0.0 and float(L.speed(0.0, steady)) == 0.0))
    v = L.knot_speeds(10.0, 14.0, 6.0)
    # hand calculation: trapezoids (0+10)/2*2 = 10, (10+14)/2*2 = 24, (14+6)/2*2 = 20 -> 10, 34, 54
    d = L.distance_at_knots(v)
    checks.append(("worked example: distances at knots are 0, 10, 34, 54 m", np.allclose(d, [0, 10, 34, 54])))
    checks.append(("distance(t) agrees with the knot table", np.allclose(L.distance(L.KNOT_T, v), d)))
    # middle of a slice: t = 1 s, speed 5 m/s, area of a triangle 0.5 * 1 * 5 = 2.5 m
    checks.append(("half-way through the first slice: 2.5 m", abs(float(L.distance(1.0, v)) - 2.5) < 1e-9))
    checks.append(("speed at t = 3 s is 12 m/s", abs(float(L.speed(3.0, v)) - 12.0) < 1e-9))
    checks.append(("slope of the distance graph at t = 3 s is 12 m/s", abs(L.slope_of_distance(3.0, v) - 12.0) < 1e-6))
    checks.append(("slope of the distance graph at t = 1 s is 5 m/s", abs(L.slope_of_distance(1.0, v) - 5.0) < 1e-6))
    # edge: car that stops has zero slope and distance stops growing
    stop = L.knot_speeds(10.0, 0.0, 0.0)
    checks.append(("a stopped car adds no distance", abs(float(L.distance(6.0, stop)) - float(L.distance(4.0, stop))) < 1e-9))
    checks.append(("distance never decreases for non-negative speeds", bool(np.all(np.diff(L.distance(np.linspace(0, 6, 61), v)) >= 0))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
